# California Yoga Studio Reviews
## Notebook 2: Review Data Preparation
*2026-05*

This notebook downloads the full California Google Reviews file,
filters it to the 423 yoga studios identified in Notebook 1, removes
duplicates, inspects review quality, and saves the cleaned review dataset
for sentiment analysis and topic modelling.

**Input:** `yoga_studios_meta.pkl`, `yoga_gmap_ids.pkl`
**Output:** `yoga_reviews.pkl` — 13,938 reviews across 423 yoga studios

## Setup

In [ ]:
# imports
from google.colab import drive
import os
import urllib.request
from tqdm import tqdm
import pickle
import pandas as pd
import gzip
import json


In [ ]:
# mounting the google drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
# setting-up directories
PROJECT_DIR = '/content/gdrive/MyDrive/YogaStudioReviews'
YOGA_META_PATH = os.path.join(PROJECT_DIR, 'yoga_studios_meta.pkl')
REVIEWS_URL = 'https://mcauleylab.ucsd.edu/public_datasets/gdrive/googlelocal/review-California.json.gz'
REVIEWS_PATH = os.path.join(PROJECT_DIR, 'review-California.json.gz')

## Download review data

In [ ]:
# downloading the review data using tqdm progress bar
class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

print('Starting download...')
with DownloadProgressBar(unit='B', unit_scale=True,
                         unit_divisor=1024, miniters=1,
                         desc='review-California.json.gz') as t:
    urllib.request.urlretrieve(REVIEWS_URL, REVIEWS_PATH, reporthook=t.update_to)

print('Download complete.')

Starting download...


review-California.json.gz: 4.98GB [13:41, 6.51MB/s]                            

Download complete.


## Load studio metadata

In [ ]:
# Reload metadata and gmap_ids from yoga_metadata_prep.ipynb notebook
YOGA_META_PATH = os.path.join(PROJECT_DIR, 'yoga_studios_meta.pkl')
yoga_meta = pd.read_pickle(YOGA_META_PATH)

with open(os.path.join(PROJECT_DIR, 'yoga_gmap_ids.pkl'), 'rb') as f:
    yoga_gmap_ids = pickle.load(f)

print(f'Loaded {len(yoga_meta)} yoga studios')
print(f'Loaded {len(yoga_gmap_ids)} gmap_ids to match against')

Loaded 423 yoga studios
Loaded 423 gmap_ids to match against


## Filter reviews to yoga studios

Scan the full reviews file line by line and retain only reviews
belonging to the 423 yoga studios. A progress update is printed
every 5 million records scanned.

In [ ]:
#filtering reviews to yoga studios only
def parse(path):
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                continue

print('Filtering reviews to yoga studios...')
yoga_reviews = []

for i, record in enumerate(parse(REVIEWS_PATH)):
    if record.get('gmap_id') in yoga_gmap_ids:
        yoga_reviews.append(record)

    # Progress update every 5 million records
    if i % 5_000_000 == 0 and i > 0:
        print(f'  Scanned {i:,} records, found {len(yoga_reviews):,} yoga reviews so far...')

yoga_reviews_df = pd.DataFrame(yoga_reviews)
print(f'\nDone. Found {len(yoga_reviews_df):,} reviews for yoga studios.')

Filtering reviews to yoga studios...
  Scanned 5,000,000 records, found 7,330 yoga reviews so far...
  Scanned 10,000,000 records, found 8,896 yoga reviews so far...
  Scanned 15,000,000 records, found 11,761 yoga reviews so far...
  Scanned 20,000,000 records, found 12,208 yoga reviews so far...
  Scanned 25,000,000 records, found 12,683 yoga reviews so far...
  Scanned 30,000,000 records, found 13,112 yoga reviews so far...
  Scanned 35,000,000 records, found 13,245 yoga reviews so far...
  Scanned 40,000,000 records, found 13,856 yoga reviews so far...
  Scanned 45,000,000 records, found 13,856 yoga reviews so far...
  Scanned 50,000,000 records, found 13,946 yoga reviews so far...
  Scanned 55,000,000 records, found 14,024 yoga reviews so far...
  Scanned 60,000,000 records, found 14,048 yoga reviews so far...
  Scanned 65,000,000 records, found 14,085 yoga reviews so far...
  Scanned 70,000,000 records, found 14,103 yoga reviews so far...

Done. Found 14,103 reviews for yoga studi

## Inspect review structure

In [ ]:
# See what columns we have
print('Columns:', yoga_reviews_df.columns.tolist())

# Sample a few records
print('\nSample reviews:')
yoga_reviews_df.head(5)

Columns: ['user_id', 'name', 'time', 'rating', 'text', 'pics', 'resp', 'gmap_id']

Sample reviews:


,user_id,name,time,rating,text,pics,resp,gmap_id
0,108813822761482384149,Woody Cullison,1582200647980,5,I have just recently started yoga at the age o...,None,"{'time': 1582216936562, 'text': 'Thank you for...",0x80dbfbda7e279df3:0x77fb2b251fb8b757
1,111897428696273122282,Isela Owen,1583958087924,5,I have been attending yoga practices at iYoga ...,None,"{'time': 1584025505683, 'text': 'Thank you bea...",0x80dbfbda7e279df3:0x77fb2b251fb8b757
2,110360923222304762236,lucyanna tran,1583957494397,5,I discovered Benny and yoga when I was 67 and ...,None,"{'time': 1584026526904, 'text': 'Thank you Luc...",0x80dbfbda7e279df3:0x77fb2b251fb8b757
3,116266789580438497500,John Bussard,1567999437412,5,Benny is TRULY a committed mentor and guide!\n...,None,"{'time': 1568062830813, 'text': 'Thank you Joh...",0x80dbfbda7e279df3:0x77fb2b251fb8b757
4,100294998690269105673,ryan knell,1594430636073,1,"DO NOT GO HERE. The ""studio"" is her dirty gara...",None,"{'time': 1625072784467, 'text': 'Well Ryan, I ...",0x80dbfbda7e279df3:0x77fb2b251fb8b757


## Remove duplicate reviews

In [ ]:
print(f'Before: {len(yoga_reviews):,} reviews')

# Checking for and removing duplicate reviews
yoga_reviews_df = yoga_reviews_df.drop_duplicates(
    subset=['gmap_id', 'user_id', 'time'], #decided to use these columns for duplicate removal only because pic column cannot be hashed by pandas
    keep='first'
)

print(f'After: {len(yoga_reviews_df):,} reviews')
print(f'Duplicates removed: {len(yoga_reviews) - len(yoga_reviews_df):,}')

Before: 14,103 reviews
After: 13,938 reviews
Duplicates removed: 165


In [ ]:
# Check for reviews with identical text
text_duplicates = yoga_reviews_df[
    yoga_reviews_df['text'].notna() &
    yoga_reviews_df.duplicated(subset=['text'], keep=False)
]

print(f'Reviews with duplicate text: {len(text_duplicates):,}')
print(f'Unique texts that appear more than once: {text_duplicates["text"].nunique():,}')

Reviews with duplicate text: 78
Unique texts that appear more than once: 35


In [ ]:
# Sort duplicate texts by how many times they appear
text_value_counts = yoga_reviews_df['text'].value_counts()
duplicate_texts = text_value_counts[text_value_counts > 1]

print(f'Most repeated review texts:')
for text, count in duplicate_texts.head(20).items():
    preview = str(text)[:80].replace('\n', ' ')
    print(f'  {count}x  "{preview}"')

Most repeated review texts:
  8x  "Love this place!"
  3x  "Amazing!"
  3x  "Amazing"
  2x  "Pretty awesome and unique yoga spot. I kind of imagine it as the "Barrys Bootcam"
  2x  "Absolutely LOVE this place! Took a class with Rachel and her voice instructions "
  2x  "Awesome place"
  2x  "Great experience!"
  2x  "Ritual is a space that I didn't see myself in. It's not a place that I could aff"
  2x  "Love it"
  2x  "Love this studio"
  2x  "🙏🙏"
  2x  "I love this studio.  First of all, you get a full week of free classes to really"
  2x  "Love this place"
  2x  "Great studio!"
  2x  "Amazing experience"
  2x  "This is by far my favorite yoga studio. The studio is clean, friendly and fully "
  2x  "Love this spot! Jenny is an amazing teacher and there are so many nice touches, "
  2x  "I bought one month unlimited and decided to go for it.  Taking regular yoga clas"
  2x  "Great place"
  2x  "Great teaching staff, pleasant and well-equipped studios, and management that ge"


Short reviews appearing multiple times across different studios
are consistent with natural reviewer behaviour rather than bot activity.
To be thorough, a second check is performed: same user leaving identical
text across different studios would be a stronger signal of suspicious behaviour.

In [ ]:
# For longer duplicate texts, show which studios they appear on
long_dupes = ['Pretty awesome and unique yoga spot',
              'Absolutely LOVE this place',
              'Ritual is a space',
              'This is by far my favorite yoga studio',
              'Love this spot! Jenny']

for snippet in long_dupes:
    matches = yoga_reviews_df[
        yoga_reviews_df['text'].str.contains(snippet, na=False)
    ].merge(
        yoga_meta[['gmap_id', 'name']],
        on='gmap_id',
        suffixes=('_reviewer', '_studio')
    )[['name_studio', 'name_reviewer', 'gmap_id', 'user_id', 'rating']]

    print(f'\nSnippet: "{snippet[:50]}..."')
    print(matches.to_string())


Snippet: "Pretty awesome and unique yoga spot..."
              name_studio  name_reviewer                                gmap_id                user_id  rating
0          Yoga Vibe WeHo   James Truimp  0x80c2bed70ef568d9:0x41a0255935b5e85d  107262189638845558839       4
1  Red Diamond Yoga Palms  Ashley Corley  0x80c2ba349867f25b:0xa7228b70e6c916a2  111365272613618112855       4

Snippet: "Absolutely LOVE this place..."
             name_studio     name_reviewer                                gmap_id                user_id  rating
0          Fire Hot Yoga  Christina Morgan  0x80c2c65283fb6591:0x1ee52b6ac7440354  109815369812941526544       4
1  Left Coast Power Yoga      Lori Salazar   0x808f875d2809dc71:0x152b24f9602d05c  117836849765043559464       4

Snippet: "Ritual is a space..."
                           name_studio          name_reviewer                                gmap_id                user_id  rating
0                         Kinship Yoga        Tom L. Williams  0x80c2c

All longer duplicate texts are posted by different users on different studios,
confirming these are independent reviewers who happen to use similar phrasing.
No text-based deduplication is performed. The dataset remains at **13,938 reviews**.

## Dataset exploration and sanity checks

Basic checks to confirm the dataset is clean and ready for analysis.

In [ ]:
# Convert timestamp to readable date
yoga_reviews_df['date'] = pd.to_datetime(yoga_reviews_df['time'], unit='ms')
print(f'\nDate range:')
print(f'  Earliest: {yoga_reviews_df["date"].min().date()}')
print(f'  Latest: {yoga_reviews_df["date"].max().date()}')


Date range:
  Earliest: 2007-11-05
  Latest: 2021-08-29


In [ ]:
# Basic sanity checks
print(f'Total reviews: {len(yoga_reviews_df):,}')
print(f'Reviews with text: {yoga_reviews_df["text"].notna().sum():,}')
print(f'Reviews without text: {yoga_reviews_df["text"].isna().sum():,}')
print(f'\nRating distribution:')
print(yoga_reviews_df['rating'].value_counts().sort_index())

Total reviews: 13,938
Reviews with text: 10,547
Reviews without text: 3,391

Rating distribution:
rating
1      311
2       96
3      154
4      772
5    12605
Name: count, dtype: int64


In [ ]:
#reviews without ratings
print(f'Reviews with no rating: {yoga_reviews_df["rating"].isna().sum()}')

Reviews with no rating: 0


In [ ]:
# reviews without text by rating
print('Rating distribution for text-less reviews:')
print(yoga_reviews_df[yoga_reviews_df['text'].isna()]['rating'].value_counts().sort_index())

Rating distribution for text-less reviews:
rating
1      66
2      17
3      46
4     162
5    3100
Name: count, dtype: int64


As expected, the  majority of text-less reviews are 5-star — satisfied
customers often do not bother writing a review if they are sattidfied about everything.

In [ ]:
# check for short reviews that are not helpful for the LDA
yoga_reviews_df['text_length'] = yoga_reviews_df['text'].str.len()
print('Text length distribution:')
print(yoga_reviews_df['text_length'].describe().round(0))

print('\nReviews under 20 characters:')
short = yoga_reviews_df[yoga_reviews_df['text_length'] < 20]['text'].value_counts().head(10)
print(short)

Text length distribution:
count    10547.0
mean       277.0
std        287.0
min          1.0
25%        101.0
50%        200.0
75%        355.0
max       4090.0
Name: text_length, dtype: float64

Reviews under 20 characters:
text
Love this place!      8
Amazing               3
Amazing!              3
Love it               2
Amazing experience    2
Awesome place         2
Love this studio      2
Love this place       2
Great place           2
Love this studio!     2
Name: count, dtype: int64


**Key observations:**
- 3,391 reviews (24%) have no text — these are rating-only and will be
  excluded from NLP analysis but retained for the classifier
- Mean review length of 277 characters is healthy for topic modelling
- Short reviews (under 20 characters) are just a fraction and
  will not meaningfully affect LDA
- The 90% five-star skew is extreme even by Google Reviews standards
  and will need to be addressed when building the classifier

In [ ]:
# Convert pics from list of URLs to a simple count
yoga_reviews_df['pic_count'] = yoga_reviews_df['pics'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print(f'Reviews with at least one photo: {(yoga_reviews_df["pic_count"] > 0).sum():,}')
print(f'Reviews with no photos: {(yoga_reviews_df["pic_count"] == 0).sum():,}')

# Now drop the raw pics column
yoga_reviews_df = yoga_reviews_df.drop(columns=['pics'])

Reviews with at least one photo: 257
Reviews with no photos: 13,681


The `pics` column contains lists of image URLs — not useful for text
analysis. It was thus coverted to a simple count (`pic_count`) to preserve
the signal of reviewer engagement, then the raw URL column was dropped.



## Output summary

The cleaned review dataset is saved for use in subsequent notebooks:

- **Total reviews:** 13,938 (deduplicated)
- **Reviews with text:** 10,547 — used for VADER and LDA
- **Rating-only reviews:** 3,391 — retained for classifier
- **Date range:** 2007–2021
- **Photo reviews:** 257 (1.8%) — `pic_count` column retained

In [ ]:
#saving the filtered df to my google drive
REVIEWS_SAVE_PATH = os.path.join(PROJECT_DIR, 'yoga_reviews.pkl')
yoga_reviews_df.to_pickle(REVIEWS_SAVE_PATH)
print(f'Saved {len(yoga_reviews_df):,} reviews.')
print(f'  — {yoga_reviews_df["text"].notna().sum():,} with text for NLP')
print(f'  — {yoga_reviews_df["text"].isna().sum():,} rating-only for classifier')

Saved 13,938 reviews.
  — 10,547 with text for NLP
  — 3,391 rating-only for classifier
